# 23 — Paper cross-species LPS replication (Bunne Fig. 4)

Tables and figures for `paper_crossspecies_{rat,mouse}_ood` using the **official CellOT plotting path** (`cellot/cellot_gpu/scripts/plot.py` + `cellot.utils.viz`).

| Panel | What we plot |
|-------|----------------|
| **f** | OOD bar charts: Pearson *r* of gene means (all 1000 HVGs) + MMD (50 markers) |
| **e** | Mean expression of LPS marker genes (i.i.d. vs o.o.d.) |
| **g** | Marginal densities for bimodal markers (`viz.plot_marginals`) |

Paper metrics ([Online Methods](https://www.nature.com/articles/s41592-023-01969-x)): *r* on all genes; MMD on top-50 DE genes; 10 bootstraps × 1000 cells.

Outputs: `paper_crossspecies_fig4_outputs/figures/`

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_context("talk")

BASE = Path("/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT").resolve()
ANALYSIS = BASE / "speciesOT/baseline/analysis"
sys.path.insert(0, str(ANALYSIS))

from paper_crossspecies_fig4_helpers import (
    PAPER_COLORS,
    PAPER_MARKERS_FIG4E,
    PAPER_MARKERS_FIG4G,
    MODEL_LABEL,
    MODEL_ORDER,
    TAGS,
    build_headline_table,
    ensure_plot_symlinks,
    load_dfs_for_marginals,
    load_evals_csv,
    load_extended_metrics,
    mean_expression_panel,
    repo_paths,
    summarize_eval_reps,
)

PATHS = repo_paths(BASE)
CELLOT_GPU = PATHS["cellot_gpu"]
RESULTS = PATHS["results"]
OUT = PATHS["out"]
FIG = OUT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

os.chdir(CELLOT_GPU)
sys.path.insert(0, str(CELLOT_GPU))

from cellot.utils import viz

print("cellot_gpu:", CELLOT_GPU)
print("figures:", FIG)

## 1. Summary tables

In [ ]:
headline = build_headline_table(RESULTS, ncells=1000)
headline.to_csv(OUT / "headline_metrics.csv", index=False)

ood = headline[headline["setting"] == "ood"].copy()
ood_display = ood[[
    "holdout", "model", "r_mean_all_genes", "r_mean_50_markers",
    "mmd_50_markers", "frac_gap_closed", "mmd_floor", "mmd_ceiling",
]].round(4)
print("OOD headline (ncells=1000)")
ood_display

In [ ]:
ext_rows = []
for holdout, tag in TAGS.items():
    for model in MODEL_ORDER:
        ext = load_extended_metrics(RESULTS, tag, model)
        if ext.empty:
            continue
        sub = ext[ext["ncells"] == 1000].copy()
        sub["holdout"] = holdout
        sub["model"] = MODEL_LABEL[model]
        ext_rows.append(sub)
extended = pd.concat(ext_rows, ignore_index=True) if ext_rows else pd.DataFrame()
if not extended.empty:
    extended.to_csv(OUT / "extended_metrics_nc1000.csv", index=False)
    extended[["holdout", "model", "n_markers", "mmd_model", "mmd_floor", "mmd_ceiling",
              "frac_gap_closed", "mean_js", "r2_model", "frac_r2_closed"]].round(4)

## 2. Fig. 4f — OOD performance bars (paper layout)

Top: Pearson *r* of means on **all genes**. Bottom: MMD on **50 markers** (from `evals.csv` reps).

In [ ]:
def fig4f_bar_data(headline_df: pd.DataFrame) -> pd.DataFrame:
    ood = headline_df[headline_df["setting"] == "ood"].copy()
    r_rows = ood[["holdout", "model", "r_mean_all_genes", "r_mean_all_genes_sd"]].copy()
    r_rows["metric"] = "r feature means (all genes)"
    r_rows["value"] = r_rows["r_mean_all_genes"]
    r_rows["sd"] = r_rows["r_mean_all_genes_sd"]
    m_rows = ood[["holdout", "model", "mmd_50_markers", "mmd_50_markers_sd"]].copy()
    m_rows["metric"] = "MMD (50 markers)"
    m_rows["value"] = m_rows["mmd_50_markers"]
    m_rows["sd"] = m_rows["mmd_50_markers_sd"]
    return pd.concat([r_rows, m_rows], ignore_index=True)

bar_df = fig4f_bar_data(headline)
bar_df.to_csv(OUT / "fig4f_bar_data.csv", index=False)

palette = {MODEL_LABEL[m]: PAPER_COLORS[m] for m in MODEL_ORDER}
order = [MODEL_LABEL[m] for m in MODEL_ORDER]

fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
for ax, metric in zip(axes, bar_df["metric"].unique()):
    sub = bar_df[bar_df["metric"] == metric]
    x = np.arange(len(sub["holdout"].unique()))
    width = 0.35
    for i, model in enumerate(order):
        msub = sub[sub["model"] == model].set_index("holdout").reindex(["rat", "mouse"])
        ax.bar(x + (i - 0.5) * width, msub["value"], width, yerr=msub["sd"],
               label=model, color=palette[model], capsize=4)
    ax.set_xticks(x)
    ax.set_xticklabels(["Rat holdout", "Mouse holdout"])
    ax.set_title(metric)
    ax.set_ylabel(metric.split("(")[0].strip())
    if metric.startswith("MMD"):
        ax.set_ylim(0, max(0.35, sub["value"].max() + 0.05))
    else:
        ax.set_ylim(0, 1.0)
axes[0].legend(loc="lower right")
fig.suptitle("Fig. 4f replica — OOD cross-species LPS (our replication)", y=1.02)
fig.tight_layout()
fig.savefig(FIG / "fig4f_ood_bars.pdf", bbox_inches="tight")
fig.savefig(FIG / "fig4f_ood_bars.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Fig. 4e — mean marker gene expression (CellOT vs scGen)

In [ ]:
def plot_fig4e(tag_name: str, holdout: str, setting: str = "ood"):
    tag_dir = RESULTS / tag_name
    panel = mean_expression_panel(tag_dir, PAPER_MARKERS_FIG4E, setting=setting)
    panel = panel[panel["layer"].isin(["treated", "cellot", "scgen"])]
    panel["layer"] = panel["layer"].replace({"cellot": "CellOT", "scgen": "scGen", "treated": "Treated"})
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.pointplot(data=panel, x="gene", y="mean_expr", hue="layer",
                  palette={"Treated": PAPER_COLORS["treated"],
                           "CellOT": PAPER_COLORS["cellot"],
                           "scGen": PAPER_COLORS["scgen"]},
                  dodge=True, ax=ax)
    ax.set_title(f"Fig. 4e replica — {holdout} holdout, {setting}")
    ax.set_xlabel("Gene")
    ax.set_ylabel("Mean expression")
    plt.xticks(rotation=45, ha="right")
    fig.tight_layout()
    out = FIG / f"fig4e_{holdout}_{setting}.pdf"
    fig.savefig(out, bbox_inches="tight")
    fig.savefig(out.with_suffix(".png"), dpi=150, bbox_inches="tight")
    plt.show()

for holdout, tag in TAGS.items():
    plot_fig4e(tag, holdout, "ood")

## 4. Fig. 4g — marginal densities (`viz.plot_marginals`)

Uses `scripts/plot.py` data loading (`model-cellot` / `model-scgen` symlinks) and the published color order.

In [ ]:
def plot_fig4g(tag_name: str, holdout: str, model_key: str = "impact_cellot"):
    tag_dir = RESULTS / tag_name
    ensure_plot_symlinks(tag_dir)
    dfs = load_dfs_for_marginals(tag_dir, model_key, setting="ood", n_markers=50)
    plot_key = "cellot" if model_key == "impact_cellot" else model_key
    plot_dfs = {
        plot_key: dfs[plot_key],
        "treated": dfs["treated"],
        "control": dfs["control"],
    }
    genes = [g for g in PAPER_MARKERS_FIG4G if g in dfs["treated"].columns]
    colors = {
        plot_key: PAPER_COLORS["cellot"],
        "treated": PAPER_COLORS["treated"],
        "control": PAPER_COLORS["control"],
    }
    order = [plot_key, "treated", "control"]
    fig, axes = viz.create_axes_grid(len(genes), min(4, len(genes)), scale=1.4)
    _, _, handles = viz.plot_marginals(
        plot_dfs,
        features=genes,
        qclip=0.01,
        colors=colors,
        order=order,
        axes=axes,
    )
    axes[0, -1].legend(handles=handles, bbox_to_anchor=(1.05, 1), loc="upper left")
    label = MODEL_LABEL[model_key]
    fig.suptitle(f"Fig. 4g replica — {holdout} OOD marginals ({label})", y=1.02)
    fig.tight_layout()
    out = FIG / f"fig4g_{holdout}_{model_key}_marginals.pdf"
    fig.savefig(out, bbox_inches="tight")
    fig.savefig(out.with_suffix(".png"), dpi=150, bbox_inches="tight")
    plt.show()

for holdout, tag in TAGS.items():
    plot_fig4g(tag, holdout, "impact_cellot")
    plot_fig4g(tag, holdout, "scgen")

## 5. Canonical PDFs via `scripts/plot.py` (optional batch export)

In [ ]:
from scripts.plot import plot_marginals, get_dfs

config_plotting = {
    "marginals": {
        "models": ["cellot", "scgen"],
        "colors": PAPER_COLORS,
    }
}

for holdout, tag in TAGS.items():
    tag_dir = RESULTS / tag
    ensure_plot_symlinks(tag_dir)
    plot_out = FIG / f"plotpy_{holdout}_ood"
    plot_out.mkdir(parents=True, exist_ok=True)
    dfs = get_dfs(tag_dir, config_plotting, setting="ood", where="data_space", n_markers=50)
    plot_marginals(config_plotting, dfs, plot_out, logscale=False)
    print(f"Wrote {plot_out / 'marginals.pdf'}")